# Linear Models

<img src="https://s44783.pcdn.co/in/wp-content/uploads/sites/3/2023/03/types-of-machine-learning.jpg.webp">

# Linear Models for Regression

Linear regression is a fundamental statistical method used to model the relationship between a dependent variable and one or more independent variables. The goal of linear regression is to find the best-fitting line (or hyperplane in higher dimensions) that describes how the dependent variable changes as the independent variables change.

<figure>
    <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/b/b0/Linear_least_squares_example2.svg/1280px-Linear_least_squares_example2.svg.png"/>
    <figcaption>Example of a linear regression model fitting a line to data points.</figcaption>
</figure>

The linear regression model can be expressed mathematically as:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \ldots + \beta_n x_n + \epsilon$$

Where:
- $\hat{y}$ is the predicted value of the dependent variable.
- $x_1, x_2, \ldots, x_n$ are the independent variables (the features or predictors).
- $\beta_0$ is the intercept (the value of $\hat{y}$ when all $x$'s are zero).
- $\beta_1, \beta_2, \ldots, \beta_n$ are the coefficients (the weights that represent the influence of each independent variable on the dependent variable).
- $\epsilon$ is the error term (the difference between the observed and predicted values).

The model determines the coefficients $\beta$ by minimizing the error/cost function which is typically the sum of squared residuals (the differences between observed and predicted values). This method is known as Ordinary Least Squares (OLS).

$$ \text{E} = \sum_{i=1}^{m} (y_i - \hat{y}_i)^2 $$

Where:
- $m$ is the number of observations.
- $y_i$ is the actual value of the dependent variable for the $i$-th observation.
- $\hat{y}_i$ is the predicted value of the dependent variable for the $i$-th observation.

## Model Assumptions

Before fitting a linear regression model, it is important to check that the data meets certain assumptions. These assumptions ensure that the model is valid and that the results can be interpreted correctly. The key assumptions of linear regression are:

1. **Linearity**: The relationship between the independent variables and the dependent variable is linear.
2. **Independence**: The residuals (errors) are independent of each other.
3. **Homoscedasticity**: The residuals have constant variance at every level of the independent variables.
4. **Normality**: The residuals are normally distributed (especially important for inference).
5. **No Multicollinearity**: The independent variables are not highly correlated with each other.


We will use the California housing dataset to demonstrate linear regression. This dataset contains information about various features of houses in California, such as median income, average house age, and average number of rooms, along with the target variable which is the median house value.

Data Dictionary:
| Feature Name | Description | Datatype |
|--------------|-------------| -------------|
| median_income       | Median income in block group | numeric |
| housing_median_age     | Median house age in block group | numeric |
| total_rooms     | Average number of rooms per household | numeric |
| total_bedrooms    | Average number of bedrooms per household | numeric |
| population    | Block group population | numeric |
| households     | Average number of household members | numeric |
| latitude     | Block group latitude | numeric |
| longitude    | Block group longitude | numeric |
| ocean_proximity | Proximity to the ocean | categorical |
| median_house_value   | Median house value for California districts (in $100,000's) | numeric */




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (MinMaxScaler,
                                   OneHotEncoder)
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import (KFold,
                                     cross_validate)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer

import math
import warnings

warnings.filterwarnings("ignore")

SEED = 67

## Load Dataset

In [ ]:
filepath = "../data/housing.csv"

df = pd.read_csv(filepath)

df.head()

## Inspect Dataset

In [ ]:
df.info()

In [ ]:
df.describe()

## Train-Test Split

Suppose you’ve chatted with some experts who told you that the median income is a
very important attribute to predict median housing prices. You may want to ensure
that the test set is representative of the various categories of incomes in the whole
dataset. Since the median income is a continuous numerical attribute, you first need
to create an income category attribute.

In [ ]:
df["income_cat"] = pd.cut(
    df["median_income"],
    bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5]
)

In [ ]:
df.head()

In [ ]:
TARGET = "median_house_value"

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    stratify=df["income_cat"],
    random_state=SEED,
)

print("Stratified income category distribution in training set:")
print(X_train["income_cat"].value_counts(normalize=True))
print("\nStratified income category distribution in test set:")
print(X_test["income_cat"].value_counts(normalize=True))

# Drop the income_cat column after stratification
X_train = X_train.drop(columns=["income_cat"])
X_test = X_test.drop(columns=["income_cat"])

# Check the shapes of the splits
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

## Exploratory Data Analysis (EDA)

In [ ]:
# check for null values in the training set
X_train.isnull().sum()

In [ ]:
categorical_cols = X_train.select_dtypes(include=["object"]).columns
numerical_cols = X_train.select_dtypes(include=["number"]).columns
print(f"Categorical columns: {categorical_cols.tolist()}")
print(f"Numerical columns: {numerical_cols.tolist()}")

In [ ]:
# plot histogram of the numerical features and target variable
hist_df = X_train[numerical_cols].copy()
hist_df[TARGET] = y_train

plt.figure(figsize=(20, 15))
for i, column in enumerate(hist_df.columns):
    plt.subplot(3, 3, i + 1)
    sns.histplot(hist_df[column], kde=True)
    plt.title(f"Distribution of {column}")
    plt.xlabel(column)
    plt.ylabel("Frequency")
plt.suptitle("Histograms of Numerical Features and Target Variable", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap of Numerical Features to Target
corr_df = X_train[numerical_cols].copy()
corr_df[TARGET] = y_train

corr_matrix = corr_df.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    mask=mask,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
)
plt.title("California Housing: Correlation Heatmap of Numerical Features")
plt.tight_layout()
plt.show()

In [ ]:
# countplot for categorical features
plt.figure(figsize=(10, 6))
sns.countplot(x="ocean_proximity", data=X_train)
plt.title("Count of Ocean Proximity Categories in Training Set")
plt.xlabel("Ocean Proximity")
plt.ylabel("Count")

for p in plt.gca().patches:
    plt.gca().annotate(
        f"{int(p.get_height())}",
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha="center",
        va="center",
        fontsize=12,
        color="black",
        xytext=(0, 5),
        textcoords="offset points",
    )

plt.tight_layout()
plt.show()

Since the values for Ocean Proximity are categorical, the proper handling of this feature is to create a one-hot encoding for it. This will create a new binary column for each category of Ocean Proximity, indicating whether a given row belongs to that category or not.

## Creating the Pipeline

From our EDA we will do the following to our pipeline.
1. Null values in total_bedrooms will be replaced with the median value of that column.
2. For the scaling of the numerical features, we will use MinMaxScaler to scale the features to a range between 0 and 1.
3. For the categorical feature 'Ocean Proximity', we will create a one-hot encoding.
4. Since 'households' has a very high correlation with total_bedrooms and population, we can drop it from the dataset to avoid multicollinearity issues.

In [ ]:
# Handling Null values
numerical_imputer = SimpleImputer(strategy="median")
categorical_imputer = SimpleImputer(strategy="most_frequent")

# Minmax scaler of numerical features
numerical_scaler = MinMaxScaler()

# Onehot encoding of categorical features
categorical_encoder = OneHotEncoder(handle_unknown="ignore")

# Creating the Pipeline
numerical_pipeline = Pipeline(
    [
        ("imputer", numerical_imputer),
        ("scaler", numerical_scaler),
    ]
)

categorical_pipeline = Pipeline(
    [
        ("imputer", categorical_imputer),
        ("encoder", categorical_encoder),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_cols.drop("households")),
        ("cat", categorical_pipeline, categorical_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

model_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ]
)

# Apply K-Fold Cross Validation
kfold = KFold(n_splits=3, shuffle=True, random_state=SEED)

In [ ]:
cv_results = cross_validate(
    model_pipeline,
    X_train,
    y_train,
    cv=kfold,
    scoring=["neg_mean_squared_error", "neg_mean_absolute_error", "r2"],
    return_train_score=True,
)

In [ ]:
cv_summary = pd.DataFrame(cv_results)

print(
    "Train RMSE:", np.sqrt(-cv_summary["train_neg_mean_squared_error"]).mean()
)
print(
    "Val RMSE:  ", np.sqrt(-cv_summary["test_neg_mean_squared_error"]).mean()
)
print("Train MAE: ", -cv_summary["train_neg_mean_absolute_error"].mean())
print("Val MAE:   ", -cv_summary["test_neg_mean_absolute_error"].mean())
print("Train R2:  ", cv_summary["train_r2"].mean())
print("Val R2:    ", cv_summary["test_r2"].mean())


In [ ]:
model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_mae = mean_absolute_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)

print("Test RMSE:", test_rmse)
print("Test MAE: ", test_mae)
print("Test R2:  ", test_r2)

In [ ]:
train_rmse = np.sqrt(-cv_summary["train_neg_mean_squared_error"]).mean()
train_mse = train_rmse**2
train_mae = -cv_summary["train_neg_mean_absolute_error"].mean()
train_r2 = cv_summary["train_r2"].mean()

val_rmse = np.sqrt(-cv_summary["test_neg_mean_squared_error"]).mean()
val_mse = val_rmse**2
val_mae = -cv_summary["test_neg_mean_absolute_error"].mean()
val_r2 = cv_summary["test_r2"].mean()

linear_regression = pd.DataFrame(
    [
        {
            "Model": "Linear Regression",
            "Set": "Train",
            "MSE": train_mse,
            "RMSE": train_rmse,
            "MAE": train_mae,
            "R2": train_r2,
        },
        {
            "Model": "Linear Regression",
            "Set": "Validation",
            "MSE": val_mse,
            "RMSE": val_rmse,
            "MAE": val_mae,
            "R2": val_r2,
        },
        {
            "Model": "Linear Regression",
            "Set": "Test",
            "MSE": mean_squared_error(y_test, y_pred),
            "RMSE": test_rmse,
            "MAE": test_mae,
            "R2": test_r2,
        },
    ]
)

regression_metrics = linear_regression


In [ ]:
metrics = ["RMSE", "MAE", "R2"]
metrics_by_set = regression_metrics.set_index("Set")[metrics]
train_vals = metrics_by_set.loc["Train"]
val_vals = metrics_by_set.loc["Validation"]
test_vals = metrics_by_set.loc["Test"]

x = np.arange(len(metrics))
width = 0.22
gap = 0.03
step = width + gap

fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

bars_train_left = ax1.bar(
    x[:2] - step, train_vals[:2], width=width, color="#1f77b4", label="Train"
)
bars_val_left = ax1.bar(
    x[:2], val_vals[:2], width=width, color="#2ca02c", label="Validation"
)
bars_test_left = ax1.bar(
    x[:2] + step, test_vals[:2], width=width, color="#ff7f0e", label="Test"
)
bars_train_right = ax2.bar(
    x[2:] - step, train_vals[2:], width=width, color="#1f77b4"
)
bars_val_right = ax2.bar(x[2:], val_vals[2:], width=width, color="#2ca02c")
bars_test_right = ax2.bar(
    x[2:] + step, test_vals[2:], width=width, color="#ff7f0e"
)

ax1.set_title("Linear Regression Train vs Validation vs Test Metrics")
ax1.set_xlabel("Metric")
ax1.set_ylabel("RMSE / MAE")
ax2.set_ylabel("R2")
ax1.set_xticks(x)
ax1.set_xticklabels(metrics)

ax1.legend(title="Set", loc="upper left", bbox_to_anchor=(1.08, 1))

for bars, ax in [
    (bars_train_left, ax1),
    (bars_val_left, ax1),
    (bars_test_left, ax1),
    (bars_train_right, ax2),
    (bars_val_right, ax2),
    (bars_test_right, ax2),
]:
    for p in bars:
        height = p.get_height()
        ax.annotate(
            f"{height:.2f}",
            (p.get_x() + p.get_width() / 2, height),
            ha="center",
            va="bottom",
            fontsize=9,
        )

plt.tight_layout()
plt.show()


## Regularization Overview

Regularization in Linear Models

When we fit a linear regression model, it can sometimes memorize noise in the training data. This is called overfitting.

Regularization helps by adding a penalty for large coefficient values.

Think of it as telling the model:
"Fit the data well, but keep the model simple."

General idea:

$$\text{Cost Function} = \text{Loss} + \lambda \times \text{Penalty} $$

Where:
- Loss is typically the mean squared error (MSE) for regression.
- Penalty is a function of the coefficients (for example, L1 or L2 norm).
- $\lambda$ is the regularization strength (a hyperparameter that controls how much we penalize large coefficients).

In this section, we compare three regularized linear models:

1. Ridge Regression
2. Lasso Regression
3. ElasticNet Regression


## Lasso Regression (L1 Regularization)

Lasso adds a penalty based on the absolute value of coefficients.

$$
\min_{\beta} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} |\beta_j|
$$

Where:
- $y_i$ is the actual value for the $i$-th observation.
- $\hat{y}_i$ is the predicted value for the $i$-th observation.
- $\beta_j$ are the coefficients for the features.

The first term is the ordinary least squares loss, and the second term is the L1 penalty that encourages sparsity in the coefficients.

What makes Lasso special:

- It can force some coefficients to become exactly zero.
- This means it can automatically perform feature selection.
- You get a simpler and more interpretable model.

Good use case:

- When you suspect only some features are truly important.
- When you want a smaller set of predictors.

Possible downside:

- If many features are strongly correlated, Lasso may pick one and ignore others somewhat arbitrarily.

The hyperparameter $\alpha$ controls the strength of the penalty.
- A higher $\alpha$ means more regularization, which can lead to more coefficients being set to zero. Which reduces model complexity and can help prevent overfitting.
- Choosing the right $\alpha$ is crucial for model performance and is often done through cross-validation.


In [ ]:
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_validate

In [ ]:
lasso_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", Lasso(alpha=1.0, max_iter=10000, random_state=SEED)),
    ]
)

# Define hyperparameter grid for alpha
param_grid = {"model__alpha": list(np.logspace(-3, 3, 5))}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    lasso_pipeline,
    param_grid,
    cv=kfold,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)

print("Tuning Lasso Regression hyperparameters...")
grid_search.fit(X_train, y_train)

print(f"\nBest alpha: {grid_search.best_params_['model__alpha']}")
print(f"Best CV MSE: {-grid_search.best_score_:.4f}")

# Use the best model
best_lasso_pipeline = grid_search.best_estimator_

# Get validation metrics using cross_validate with multiple scorers
scoring = {
    "neg_mean_squared_error": "neg_mean_squared_error",
    "neg_mean_absolute_error": "neg_mean_absolute_error",
    "r2": "r2",
}

cv_results = cross_validate(
    best_lasso_pipeline, X_train, y_train, cv=3, scoring=scoring, n_jobs=-1
)

val_mse = -cv_results["test_neg_mean_squared_error"].mean()
val_mae = -cv_results["test_neg_mean_absolute_error"].mean()
val_r2 = cv_results["test_r2"].mean()
val_rmse = np.sqrt(val_mse)

# Train metrics
y_train_pred = best_lasso_pipeline.predict(X_train)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

# Test metrics
best_lasso_pipeline.fit(X_train, y_train)
y_test_pred = best_lasso_pipeline.predict(X_test)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Save metrics with all validation metrics
lasso_regression = pd.DataFrame(
    {
        "Model": ["Lasso Regression"] * 3,
        "Set": ["Train", "Validation", "Test"],
        "MSE": [train_mse, val_mse, test_mse],
        "RMSE": [train_rmse, val_rmse, test_rmse],
        "MAE": [train_mae, val_mae, test_mae],
        "R2": [train_r2, val_r2, test_r2],
    }
)

regression_metrics = pd.concat(
    [regression_metrics, lasso_regression], ignore_index=True
)

print("\nTuned Lasso Final Test Metrics:")
print(f"MSE : {test_mse:.4f}")
print(f"RMSE: {test_rmse:.4f}")
print(f"MAE : {test_mae:.4f}")
print(f"R2  : {test_r2:.4f}")

In [ ]:
regression_metrics

In [ ]:
plot_df = regression_metrics.melt(
    id_vars=["Model", "Set"],
    value_vars=["RMSE", "MAE", "R2"],
    var_name="Metric",
    value_name="Value",
)

lasso_plot_df = plot_df[plot_df["Model"] == "Lasso Regression"]
left_df = lasso_plot_df[lasso_plot_df["Metric"].isin(["RMSE", "MAE"])]
right_df = lasso_plot_df[lasso_plot_df["Metric"] == "R2"]

fig, ax1 = plt.subplots(figsize=(15, 6))
ax2 = ax1.twinx()

sns.barplot(data=left_df, x="Metric", y="Value", hue="Set", ax=ax1)
sns.barplot(
    data=right_df, x="Metric", y="Value", hue="Set", ax=ax2, legend=False
)

ax1.set_title("Lasso Regression: Metrics by Dataset")
ax1.set_xlabel("Metric")
ax1.set_ylabel("RMSE / MAE")
ax2.set_ylabel("R2")
ax1.legend(title="Set", loc="upper left", bbox_to_anchor=(1.08, 1))

for ax in [ax1, ax2]:
    for p in ax.patches:
        height = p.get_height()
        if height > 0.0001:
            ax.annotate(
                f"{height:.2f}",
                (p.get_x() + p.get_width() / 2.0, height),
                ha="center",
                va="bottom",
                fontsize=9,
            )

plt.tight_layout()
plt.show()


In [ ]:
test_metrics = regression_metrics[regression_metrics["Set"] == "Test"]
plot_df = test_metrics.set_index("Model")[["RMSE", "MAE", "R2"]].T

metrics = plot_df.index.tolist()
models = plot_df.columns.tolist()
n_models = len(models)

x = np.arange(len(metrics))
width = 0.8 / n_models
colors = plt.cm.tab10.colors

fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

bar_containers = []
for i, model in enumerate(models):
    offset = (i - (n_models - 1) / 2) * width
    vals = plot_df[model]
    bars_left = ax1.bar(
        x[:2] + offset,
        vals.iloc[:2],
        width=width,
        color=colors[i],
        label=model,
    )
    bars_right = ax2.bar(
        x[2:] + offset, vals.iloc[2:], width=width, color=colors[i]
    )
    bar_containers += [bars_left, bars_right]

ax1.set_title("Linear Regression Variants Test Set Metrics")
ax1.set_xlabel("Metric")
ax1.set_ylabel("RMSE / MAE")
ax2.set_ylabel("R2")
ax1.set_xticks(x)
ax1.set_xticklabels(metrics)
ax1.legend(title="Model", loc="upper left", bbox_to_anchor=(1.08, 1))

for bars in bar_containers:
    for p in bars:
        height = p.get_height()
        ax1_or_ax2 = (
            ax1 if bars in (bc for bc in bar_containers[0::2]) else ax2
        )
        p.axes.annotate(
            f"{height:.2f}",
            (p.get_x() + p.get_width() / 2, height),
            ha="center",
            va="bottom",
            fontsize=9,
        )

plt.tight_layout()
plt.show()


### Looking into Feature Importance

Feature importance is done so that we can understand which features are most influential in making predictions. This help in discussing with non-technical stakeholders and in making informed decisions about feature selection and engineering.

In linear models this is often assessed by looking at the magnitude of the coefficients. However, this can be misleading if features are on different scales or if there is multicollinearity.

In [ ]:
from sklearn.inspection import permutation_importance

What permutation importance does is it randomly shuffles the values of a feature and measures how much the model's performance decreases. If the performance drops significantly, it means that feature is important for the model's predictions. This method is model-agnostic and can be applied to any fitted model, making it a powerful tool for understanding feature importance in linear regression and beyond.

In [ ]:
perm_importance = permutation_importance(
    best_lasso_pipeline,
    X_test,
    y_test,
    n_repeats=10,
    random_state=SEED,
    n_jobs=-1,
)

perm_importance_df = pd.DataFrame(
    {
        "Feature": X_train.columns,
        "Importance": perm_importance.importances_mean,
    }
).sort_values("Importance", ascending=True)

plt.figure(figsize=(12, 6))
plt.barh(
    perm_importance_df["Feature"],
    perm_importance_df["Importance"],
    color="coral",
)

for index, value in enumerate(perm_importance_df["Importance"]):
    plt.text(value, index, f"{value:.4f}", va="center")

plt.xlabel("Permutation Importance")
plt.title("Lasso Regression - Permutation Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    X_train["longitude"],
    X_train["latitude"],
    c=y_train,
    cmap="viridis",
    alpha=0.5,
    s=20,
)
plt.colorbar(scatter, label="Median House Value")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("California Housing: Geographic Distribution of Target Variable")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## Ridge Regression (L2 Regularization)

Ridge adds a penalty based on the squared size of coefficients.

$$
\min_{\beta} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p}\beta_j^2
$$

Where:
- $y_i$ is the actual value for the $i$-th observation.
- $\hat{y}_i$ is the predicted value for the $i$-th observation.
- $\beta_j$ are the coefficients for the features.

The first term is the ordinary least squares loss, and the second term is the L2 penalty that encourages smaller coefficients but does not set them to zero.

What this means:

- The model still uses all features.
- Coefficients are pushed closer to zero, but usually not exactly zero.
- It is very useful when features are highly correlated (multicollinearity).

Why this matters for our housing data:

- Features like Latitude and Longitude can be strongly related.
- Ridge makes coefficient estimates more stable in this situation.

How to interpret alpha:

- Larger alpha means stronger penalty, simpler model, potentially less overfitting.
- Smaller alpha means behavior closer to ordinary linear regression.

The hyperparameter $\alpha$ controls the strength of the penalty.
- A higher $\alpha$ means more regularization, which can lead to smaller coefficients and a simpler model. This can help prevent overfitting, especially when there are many features or multicollinearity.
- Choosing the right $\alpha$ is crucial for model performance and is often done through cross-validation.

In [ ]:
ridge_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", Ridge(alpha=1.0, max_iter=10000, random_state=SEED)),
    ]
)

# Define hyperparameter grid for alpha
param_grid = {"model__alpha": list(np.logspace(-3, 3, 5))}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    ridge_pipeline,
    param_grid,
    cv=kfold,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)

print("Tuning Ridge Regression hyperparameters...")
grid_search.fit(X_train, y_train)

print(f"\nBest alpha: {grid_search.best_params_['model__alpha']}")
print(f"Best CV MSE: {-grid_search.best_score_:.4f}")

# Use the best model
best_ridge_pipeline = grid_search.best_estimator_

# Get validation metrics using cross_validate with multiple scorers
scoring = {
    "neg_mean_squared_error": "neg_mean_squared_error",
    "neg_mean_absolute_error": "neg_mean_absolute_error",
    "r2": "r2",
}

cv_results = cross_validate(
    best_ridge_pipeline, X_train, y_train, cv=3, scoring=scoring, n_jobs=-1
)

val_mse = -cv_results["test_neg_mean_squared_error"].mean()
val_mae = -cv_results["test_neg_mean_absolute_error"].mean()
val_r2 = cv_results["test_r2"].mean()
val_rmse = np.sqrt(val_mse)

# Train metrics
y_train_pred = best_ridge_pipeline.predict(X_train)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

# Test metrics
best_ridge_pipeline.fit(X_train, y_train)
y_test_pred = best_ridge_pipeline.predict(X_test)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Save metrics with all validation metrics
ridge_regression = pd.DataFrame(
    {
        "Model": ["Ridge Regression"] * 3,
        "Set": ["Train", "Validation", "Test"],
        "MSE": [train_mse, val_mse, test_mse],
        "RMSE": [train_rmse, val_rmse, test_rmse],
        "MAE": [train_mae, val_mae, test_mae],
        "R2": [train_r2, val_r2, test_r2],
    }
)

regression_metrics = pd.concat(
    [regression_metrics, ridge_regression], ignore_index=True
)

print("\nTuned Ridge Final Test Metrics:")
print(f"MSE : {test_mse:.4f}")
print(f"RMSE: {test_rmse:.4f}")
print(f"MAE : {test_mae:.4f}")
print(f"R2  : {test_r2:.4f}")


In [ ]:
regression_metrics

In [ ]:
plot_df = regression_metrics.melt(
    id_vars=["Model", "Set"],
    value_vars=["RMSE", "MAE", "R2"],
    var_name="Metric",
    value_name="Value",
)

ridge_plot_df = plot_df[plot_df["Model"] == "Ridge Regression"]
left_df = ridge_plot_df[ridge_plot_df["Metric"].isin(["RMSE", "MAE"])]
right_df = ridge_plot_df[ridge_plot_df["Metric"] == "R2"]

fig, ax1 = plt.subplots(figsize=(15, 6))
ax2 = ax1.twinx()

sns.barplot(data=left_df, x="Metric", y="Value", hue="Set", ax=ax1)
sns.barplot(
    data=right_df, x="Metric", y="Value", hue="Set", ax=ax2, legend=False
)

ax1.set_title("Ridge Regression: Metrics by Dataset")
ax1.set_xlabel("Metric")
ax1.set_ylabel("RMSE / MAE")
ax2.set_ylabel("R2")
ax1.legend(title="Set", loc="upper left", bbox_to_anchor=(1.08, 1))

for ax in [ax1, ax2]:
    for p in ax.patches:
        height = p.get_height()
        if height > 0.0001:
            ax.annotate(
                f"{height:.2f}",
                (p.get_x() + p.get_width() / 2.0, height),
                ha="center",
                va="bottom",
                fontsize=9,
            )

plt.tight_layout()
plt.show()


In [ ]:
test_metrics = regression_metrics[regression_metrics["Set"] == "Test"]
plot_df = test_metrics.set_index("Model")[["RMSE", "MAE", "R2"]].T

metrics = plot_df.index.tolist()
models = plot_df.columns.tolist()
n_models = len(models)

x = np.arange(len(metrics))
width = 0.8 / n_models
colors = plt.cm.tab10.colors

fig, ax1 = plt.subplots(figsize=(15, 6))
ax2 = ax1.twinx()

bar_containers = []
for i, model in enumerate(models):
    offset = (i - (n_models - 1) / 2) * width
    vals = plot_df[model]
    bars_left = ax1.bar(
        x[:2] + offset,
        vals.iloc[:2],
        width=width,
        color=colors[i],
        label=model,
    )
    bars_right = ax2.bar(
        x[2:] + offset, vals.iloc[2:], width=width, color=colors[i]
    )
    bar_containers += [bars_left, bars_right]

ax1.set_title("Linear Regression Variants Test Set Metrics")
ax1.set_xlabel("Metric")
ax1.set_ylabel("RMSE / MAE")
ax2.set_ylabel("R2")
ax1.set_xticks(x)
ax1.set_xticklabels(metrics)
ax1.legend(title="Model", loc="upper left", bbox_to_anchor=(1.08, 1))

for bars in bar_containers:
    for p in bars:
        height = p.get_height()
        ax1_or_ax2 = (
            ax1 if bars in (bc for bc in bar_containers[0::2]) else ax2
        )
        p.axes.annotate(
            f"{height:.2f}",
            (p.get_x() + p.get_width() / 2, height),
            ha="center",
            va="bottom",
            fontsize=9,
        )

plt.tight_layout()
plt.show()


In [ ]:
perm_importance = permutation_importance(
    best_ridge_pipeline,
    X_test,
    y_test,
    n_repeats=10,
    random_state=SEED,
    n_jobs=-1,
)

perm_importance_df = pd.DataFrame(
    {
        "Feature": X_train.columns,
        "Importance": perm_importance.importances_mean,
    }
).sort_values("Importance", ascending=True)

plt.figure(figsize=(12, 6))
plt.barh(
    perm_importance_df["Feature"],
    perm_importance_df["Importance"],
    color="coral",
)

for index, value in enumerate(perm_importance_df["Importance"]):
    plt.text(value, index, f"{value:.4f}", va="center")

plt.xlabel("Permutation Importance")
plt.title("Ridge Regression - Permutation Feature Importance")
plt.tight_layout()
plt.show()

## Comparing with KNN Regression

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
knn_reg_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", KNeighborsRegressor(n_neighbors=20)
        ),
    ]
)

# Define hyperparameter grid for n_neighbors
param_grid = {"model__n_neighbors": [3, 5, 7, 9]}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    knn_reg_pipeline,
    param_grid,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)

print("Tuning KNN Regression hyperparameters...")
grid_search.fit(X_train, y_train)

print(
    f"\nBest n_neighbors: {grid_search.best_params_['model__n_neighbors']}"
)
print(f"Best CV MSE: {-grid_search.best_score_:.4f}")

# Use the best model
best_knn_pipeline = grid_search.best_estimator_

# Get validation metrics using cross_validate with multiple scorers
scoring = {
    "neg_mean_squared_error": "neg_mean_squared_error",
    "neg_mean_absolute_error": "neg_mean_absolute_error",
    "r2": "r2",
}

cv_results = cross_validate(
    best_knn_pipeline, X_train, y_train, cv=3, scoring=scoring, n_jobs=-1
)

val_mse = -cv_results["test_neg_mean_squared_error"].mean()
val_mae = -cv_results["test_neg_mean_absolute_error"].mean()
val_r2 = cv_results["test_r2"].mean()
val_rmse = np.sqrt(val_mse)

# Train metrics
y_train_pred = best_knn_pipeline.predict(X_train)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

# Test metrics
best_knn_pipeline.fit(X_train, y_train)
y_test_pred = best_knn_pipeline.predict(X_test)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Save metrics with all validation metrics
knn_regression = pd.DataFrame(
    {
        "Model": ["KNN Regression"] * 3,
        "Set": ["Train", "Validation", "Test"],
        "MSE": [train_mse, val_mse, test_mse],
        "RMSE": [train_rmse, val_rmse, test_rmse],
        "MAE": [train_mae, val_mae, test_mae],
        "R2": [train_r2, val_r2, test_r2],
    }
)

regression_metrics = pd.concat(
    [regression_metrics, knn_regression], ignore_index=True
)

print("\nTuned KNN Final Test Metrics:")
print(f"MSE : {test_mse:.4f}")
print(f"RMSE: {test_rmse:.4f}")
print(f"MAE : {test_mae:.4f}")
print(f"R2  : {test_r2:.4f}")


In [ ]:
regression_metrics

In [ ]:
plot_df = regression_metrics.melt(
    id_vars=["Model", "Set"],
    value_vars=["RMSE", "MAE", "R2"],
    var_name="Metric",
    value_name="Value",
)

ridge_plot_df = plot_df[plot_df["Model"] == "KNN Regression"]
left_df = ridge_plot_df[ridge_plot_df["Metric"].isin(["RMSE", "MAE"])]
right_df = ridge_plot_df[ridge_plot_df["Metric"] == "R2"]

fig, ax1 = plt.subplots(figsize=(15, 6))
ax2 = ax1.twinx()

sns.barplot(data=left_df, x="Metric", y="Value", hue="Set", ax=ax1)
sns.barplot(
    data=right_df, x="Metric", y="Value", hue="Set", ax=ax2, legend=False
)

ax1.set_title("Ridge Regression: Metrics by Dataset")
ax1.set_xlabel("Metric")
ax1.set_ylabel("RMSE / MAE")
ax2.set_ylabel("R2")
ax1.legend(title="Set", loc="upper left", bbox_to_anchor=(1.08, 1))

for ax in [ax1, ax2]:
    for p in ax.patches:
        height = p.get_height()
        if height > 0.0001:
            ax.annotate(
                f"{height:.2f}",
                (p.get_x() + p.get_width() / 2.0, height),
                ha="center",
                va="bottom",
                fontsize=9,
            )

plt.tight_layout()
plt.show()


In [ ]:
test_metrics = regression_metrics[regression_metrics["Set"] == "Test"]
plot_df = test_metrics.set_index("Model")[["RMSE", "MAE", "R2"]].T

metrics = plot_df.index.tolist()
models = plot_df.columns.tolist()
n_models = len(models)

x = np.arange(len(metrics))
width = 0.8 / n_models
colors = plt.cm.tab10.colors

fig, ax1 = plt.subplots(figsize=(15, 6))
ax2 = ax1.twinx()

bar_containers = []
for i, model in enumerate(models):
    offset = (i - (n_models - 1) / 2) * width
    vals = plot_df[model]
    bars_left = ax1.bar(
        x[:2] + offset,
        vals.iloc[:2],
        width=width,
        color=colors[i],
        label=model,
    )
    bars_right = ax2.bar(
        x[2:] + offset, vals.iloc[2:], width=width, color=colors[i]
    )
    bar_containers += [bars_left, bars_right]

ax1.set_title("Linear Regression Variants Test Set Metrics")
ax1.set_xlabel("Metric")
ax1.set_ylabel("RMSE / MAE")
ax2.set_ylabel("R2")
ax1.set_xticks(x)
ax1.set_xticklabels(metrics)
ax1.legend(title="Model", loc="upper left", bbox_to_anchor=(1.08, 1))

for bars in bar_containers:
    for p in bars:
        height = p.get_height()
        ax1_or_ax2 = (
            ax1 if bars in (bc for bc in bar_containers[0::2]) else ax2
        )
        p.axes.annotate(
            f"{height:.2f}",
            (p.get_x() + p.get_width() / 2, height),
            ha="center",
            va="bottom",
            fontsize=9,
        )

plt.tight_layout()
plt.show()

In [ ]:
# reset jupyter variables
%reset -f

# Linear Models for Classification

Linear models are also extensively used for classification. Let’s look at binary classification first. In this case, a prediction is made using the following formula:

\begin{align*}
&\hat{y} = w_0 x_0 + w_1 x_1 + \dotsc + w_p x_p + b > \text{threshold}
\end{align*}

The formula looks very similar to the one for linear regression, but instead of just returning the weighted sum of the features, we threshold the predicted value (say at zero). If the function was smaller than zero, we predict the class -1, if it was larger than zero, we predict the class +1.  This prediction rule is common to all linear models for classification. Again, there are many different ways to find the coefficients $\{w[i]\}$ and the intercept $b$.

For linear models for regression, the output $\hat{y}$ was a linear function of the features: a
line, plane, or hyperplane (in higher dimensions). For linear models for classification,
the decision boundary is a linear function of the input. In other words, a (binary) linear
classifier is a classifier that separates two classes using a line, a plane or a hyperplane. We will see examples of that below.

There are many algorithms for learning linear models. These algorithms all differ in the following two ways:

**1.** How they measure how well a particular **combination** of coefficients and intercept fits the training data.

**2.** If and what kind of **regularization** they use.

# Logistic Regression

Logistic Regression is a statistical method used for binary classification problems. It models the probability that a given input belongs to a particular class. The logistic regression model is based on the logistic function, which maps any real-valued number into the (0, 1) interval, making it suitable for modeling probabilities.

Though it is called "regression," it is actually a classification algorithm. The output of logistic regression is a probability that the input belongs to the positive class, and it can be used to make binary predictions by applying a threshold (commonly 0.5).

Mathematically, the logistic regression the component of $\hat{y}$ is calibrated by assuming that the state of $\hat{y}$ can be separated into say two states [0 (fail) and 1 (success)] using a logistic or a sigmoid function given by:

\begin{equation}
p(z_i) = p(w_i, x_i) = \frac{1}{1 + e^{-\hat{y} (w_i, x_i)}}
\end{equation}

Where: $\hat{y}=\sum_i(w_i x_i) + b$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


# Input range and output probabilities
z = np.linspace(-10, 10, 400)
p = sigmoid(z)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(
    z, p, color="teal", linewidth=2, label=r"$\sigma(z)=\frac{1}{1+e^{-z}}$"
)
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1)
plt.axvline(0, color="gray", linestyle="--", linewidth=1)

plt.title("Sigmoid Curve (Logistic Regression)")
plt.xlabel("Linear score (z = w·x + b)")
plt.ylabel("Predicted probability P(y=1)")
plt.ylim(-0.05, 1.05)
plt.grid(alpha=0.25)
plt.legend()
plt.show()

The goal of logistic regression is to find the best-fitting coefficients $w_i$ and intercept $b$ that maximize the likelihood of the observed data. This is typically done using maximum likelihood estimation (MLE). The cost function for logistic regression is based on the likelihood of the observed data, and it is often optimized using algorithms like gradient descent or more advanced optimization techniques.

Model Assumptions:
1. **Linearity of the Logit**: The log odds of the outcome is a linear combination of the independent variables.
2. **Independence of Errors**: The observations are independent of each other.
3. **No Multicollinearity**: The independent variables are not highly correlated with each other.
4. **Large Sample Size**: Logistic regression requires a large sample size to provide reliable estimates, especially when there are many predictors.

The hyperparameter of Logistic Regression is the regularization strength (C) and the type of regularization (L1, L2, or ElasticNet). Regularization helps prevent overfitting by adding a penalty to the cost function based on the magnitude of the coefficients. The choice of regularization and its strength can significantly impact the performance of the logistic regression model. Higher values of C reduces overfitting by increasing the penalty for large coefficients, while lower values of C allow for more flexibility in fitting the training data. Choosing the right value of C is crucial and is often done through cross-validation.

We will use the Boston breast cancer dataset to demonstrate logistic regression. This dataset contains features computed from breast mass images, and the target variable indicates whether the mass is malignant (1) or benign (0).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_validate,
)
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    recall_score,
    precision_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.inspection import permutation_importance


SEED = 67

## Load Dataset

In [ ]:
df = pd.DataFrame(
    load_breast_cancer().data, columns=load_breast_cancer().feature_names
)
df["target"] = load_breast_cancer().target

# Flip the target so that 1 = malignant and 0 = benign to make it more intuitive
df["target"] = 1 - df["target"]

df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
# plot bar plot of target variable
plt.figure(figsize=(6, 6))
sns.countplot(x="target", data=df, hue="target")

for p in plt.gca().patches:
    height = p.get_height()
    if height > 0:
        plt.gca().annotate(
            f"{height}",
            (p.get_x() + p.get_width() / 2, height),
            ha="center",
            va="bottom",
            fontsize=10,
        )

plt.title("Distribution of Target Variable")
plt.xlabel("Target Class")
plt.ylabel("Count")
plt.xticks([0, 1], ["Benign (0)", "Malignant (1)"])
plt.tight_layout()
plt.show()

Check the PCC

In [ ]:
# PCC Calculation
class_counts = df["target"].value_counts()
class_proportions = class_counts / len(df)
pcc = (class_proportions**2).sum()
print(f"Proportional Chance Criterion (PCC): {1.25 * pcc:.4f}")

## Train Test Split

In [ ]:
TARGET = "target"

X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

## Creating the Pipeline

### Baseline Logistic Regression Pipeline

In [ ]:
# Create baseline logistic regression pipeline
baseline_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=10000, random_state=SEED)),
    ]
)

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_recall, cv_precision, cv_f1, cv_roc_auc = [], [], [], []

for fold, (train_idx, val_idx) in enumerate(
    kf.split(X_train, y_train), start=1
):
    X_train_fold = X_train.iloc[train_idx]
    X_val_fold = X_train.iloc[val_idx]
    y_train_fold = y_train.iloc[train_idx]
    y_val_fold = y_train.iloc[val_idx]

    baseline_pipeline.fit(X_train_fold, y_train_fold)
    y_val_pred = baseline_pipeline.predict(X_val_fold)
    y_val_pred_proba = baseline_pipeline.predict_proba(X_val_fold)[:, 1]

    recall = recall_score(y_val_fold, y_val_pred)
    precision = precision_score(y_val_fold, y_val_pred)
    f1 = f1_score(y_val_fold, y_val_pred)

    cv_recall.append(recall)
    cv_precision.append(precision)
    cv_f1.append(f1)

    print(
        f"Fold {fold} - Recall: {recall:.4f}, Precision: {precision:.4f}, F1: {f1:.4f}"
    )

print("\nCV Averages:")
print(f"Recall  : {np.mean(cv_recall):.4f}")
print(f"Precision: {np.mean(cv_precision):.4f}")
print(f"F1      : {np.mean(cv_f1):.4f}")

# Train metrics
y_train_pred = baseline_pipeline.predict(X_train)
train_recall = recall_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred)

# Final holdout evaluation
baseline_pipeline.fit(X_train, y_train)
y_test_pred = baseline_pipeline.predict(X_test)
y_test_pred_proba = baseline_pipeline.predict_proba(X_test)[:, 1]

test_recall = recall_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)

# Save metrics in a dataframe
classification_metrics = pd.DataFrame(
    {
        "Model": ["Baseline Logistic Regression"] * 3,
        "Set": ["Train", "Validation", "Test"],
        "Recall": [train_recall, np.mean(cv_recall), test_recall],
        "Precision": [train_precision, np.mean(cv_precision), test_precision],
        "F1": [train_f1, np.mean(cv_f1), test_f1],
    }
)

print("\nFinal Holdout Test Metrics:")
print(f"Recall  : {test_recall:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"F1      : {test_f1:.4f}")

# print classification report
print("\nClassification Report on Test Set:")
print(
    classification_report(
        y_test, y_test_pred, target_names=["Benign (0)", "Malignant (1)"]
    )
)

In [ ]:
classification_metrics

In [ ]:
plot_df = classification_metrics.melt(
    id_vars=["Model", "Set"],
    value_vars=["Recall", "Precision", "F1"],
    var_name="Metric",
    value_name="Value",
)

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=plot_df, x="Metric", y="Value", hue="Set")
ax.set_title("Baseline Logistic Regression: Metrics by Dataset")
ax.set_xlabel("Metric")
ax.set_ylabel("Metric Value")

for p in ax.patches:
    height = p.get_height()
    if height > 0.0001:
        ax.annotate(
            f"{height:.4f}",
            (p.get_x() + p.get_width() / 2.0, height),
            ha="center",
            va="bottom",
            fontsize=9,
        )

plt.tight_layout()
plt.show()

#### Confusion Matrix

In [ ]:
y_test_pred = baseline_pipeline.predict(X_test)
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Baseline Logistic Regression")
plt.xticks([0.5, 1.5], ["Benign (0)", "Malignant (1)"])
plt.yticks([0.5, 1.5], ["Benign (0)", "Malignant (1)"])
plt.tight_layout()
plt.show()

### Improved Pipeline Logistic Regression Model

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "scaler",
            Pipeline(
                steps=[
                    ("scale", StandardScaler()),
                ]
            ),
            X_train.columns,
        ),
    ]
)

improved_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=50000, random_state=SEED)),
    ]
)

# Define hyperparameter grid for GridSearchCV
param_grid = [
    {
        "model__penalty": [
            "l1"
        ],  # because l1 only works with certain solvers,
        # we need to separate the grid for l1 and l2
        "model__C": [0.01, 0.1, 1, 10],
        "model__solver": ["liblinear", "saga"],
    },
    {
        "model__penalty": ["l2"],
        "model__C": [0.01, 0.1, 1, 10],
        "model__solver": ["lbfgs", "liblinear", "saga"],
    },
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(
    improved_pipeline,
    param_grid,
    cv=cv,
    scoring="f1",  # Changed to f1 for better balance
    n_jobs=-1,
    verbose=1,
)

print("Tuning Logistic Regression hyperparameters...")
grid_search.fit(X_train, y_train)

# Print results
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV F1 Score: {grid_search.best_score_:.4f}")

# Use best model for final evaluation
best_model = grid_search.best_estimator_

# Train set predictions
y_train_pred = best_model.predict(X_train)
train_recall = recall_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred)

# Test set predictions
y_test_pred = best_model.predict(X_test)
y_test_pred_proba = best_model.predict_proba(X_test)[:, 1]
test_recall = recall_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)

scoring_metrics = {"recall": "recall", "precision": "precision", "f1": "f1"}

cv_scores = cross_validate(
    best_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring_metrics,
    return_train_score=False,
)

# Extract mean validation scores
val_recall = cv_scores["test_recall"].mean()
val_precision = cv_scores["test_precision"].mean()
val_f1 = cv_scores["test_f1"].mean()

print("Validation Metrics (5-Fold CV):")
print(f"Recall  : {val_recall:.4f} (+/- {cv_scores['test_recall'].std():.4f})")
print(
    f"Precision: {val_precision:.4f} (+/- {cv_scores['test_precision'].std():.4f})"
)
print(f"F1      : {val_f1:.4f} (+/- {cv_scores['test_f1'].std():.4f})")

# Train set predictions
y_train_pred = best_model.predict(X_train)
train_recall = recall_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred)

# Test set predictions
y_test_pred = best_model.predict(X_test)
test_recall = recall_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)

# Generate classification report
print("\nFinal Test Set Metrics:")
print(f"Recall  : {test_recall:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"F1      : {test_f1:.4f}")

print("\nClassification Report on Test Set:")
print(
    classification_report(
        y_test, y_test_pred, target_names=["Malignant (0)", "Benign (1)"]
    )
)

# Save metrics in a dataframe with train, validation, and test sets
tuned_log_reg_metrics = pd.DataFrame(
    {
        "Model": ["Tuned Logistic Regression"] * 3,
        "Set": ["Train", "Validation", "Test"],
        "Recall": [train_recall, val_recall, test_recall],
        "Precision": [train_precision, val_precision, test_precision],
        "F1": [train_f1, val_f1, test_f1],
    }
)

# append to classification_metrics
classification_metrics = pd.concat(
    [classification_metrics, tuned_log_reg_metrics], ignore_index=True
)

In [ ]:
classification_metrics

In [ ]:
plot_df = classification_metrics.melt(
    id_vars=["Model", "Set"],
    value_vars=["Recall", "Precision", "F1"],
    var_name="Metric",
    value_name="Value",
)

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=plot_df, x="Metric", y="Value", hue="Set")
ax.set_title("Improved Logistic Regression: Metrics by Dataset")
ax.set_xlabel("Metric")
ax.set_ylabel("Metric Value")

for p in ax.patches:
    height = p.get_height()
    if height > 0.0001:
        ax.annotate(
            f"{height:.4f}",
            (p.get_x() + p.get_width() / 2.0, height),
            ha="center",
            va="bottom",
            fontsize=9,
        )

plt.tight_layout()
plt.show()


In [ ]:
y_test_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Malignant (0)", "Benign (1)"],
    yticklabels=["Malignant (0)", "Benign (1)"],
    cbar=False,
)
plt.title("Confusion Matrix - Best Model")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()

In [ ]:
test_metrics = classification_metrics[classification_metrics["Set"] == "Test"]
plot_df = test_metrics.set_index("Model")[["Recall", "Precision", "F1"]].T

ax = plot_df.plot(kind="bar", figsize=(15, 6), width=0.7)
ax.set_title("Logistic Regression Test Set Metrics")
ax.set_xlabel("Metric")
ax.set_ylabel("Metric Value")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Model", loc="lower right")

for p in ax.patches:
    height = p.get_height()
    ax.annotate(
        f"{height:.2f}",
        (p.get_x() + p.get_width() / 2, height),
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()


In [ ]:
perm_importance = permutation_importance(
    best_model, X_test, y_test, n_repeats=10, random_state=SEED, n_jobs=-1
)

perm_importance_df = pd.DataFrame(
    {
        "Feature": X_train.columns.tolist(),
        "Importance": perm_importance.importances_mean,
        "Std": perm_importance.importances_std,
    }
).sort_values("Importance", ascending=False)

# Plot permutation importance
fig, ax = plt.subplots(figsize=(10, 8))

top_perm = perm_importance_df.head(15)
ax.barh(range(len(top_perm)), top_perm["Importance"])
ax.set_yticks(range(len(top_perm)))
ax.set_yticklabels(top_perm["Feature"])
ax.set_xlabel("Permutation Importance (F1 Score Drop)")
ax.set_title(
    "Top 15 Features by Permutation Importance\n(How much accuracy decreases when feature is shuffled)"
)
ax.invert_yaxis()

plt.tight_layout()
plt.show()